Final Code

In [ ]:
!pip install rank_bm25

In [ ]:
import json
import pickle
import torch
import pandas as pd
import torch
from sentence_transformers.util import cos_sim
from collections import defaultdict
import re
from torch.utils.data import DataLoader
import math
from rank_bm25 import BM25Okapi

In [ ]:
!pip install rank_bm25


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
!pip install sentence-transformers

In [ ]:
with open("chunks_index.json", "r") as f:
    sec_chunks = json.load(f)


df = pd.read_csv("finder_augmented.csv")

df["company_name"] = df["company_name"].astype(str)

finder_queries = df.to_dict(orient="records")

finder_companies = set(df["company_name"].dropna().unique())
print(f"Found {len(finder_companies)} companies in FINdER dataset:", finder_companies)

filtered_chunks = [ch for ch in sec_chunks if ch["filename"] in finder_companies]

corpus_texts = [c["text"] for c in filtered_chunks]


def build_bm25(corpus_texts, tokenizer):
    tokenized = [tokenizer(t) for t in corpus_texts]
    return BM25Okapi(tokenized)

def normalize(s):
    return re.sub(r"\s+", " ", str(s).strip())
def tokenize(s):
    return normalize(s).lower().split()

bm25 = build_bm25(corpus_texts, tokenize)


def rrf_retrieve(query, dense_model, corpus_embeddings, corpus_texts, bm25, dense_k=200,bm25_k=200,rrf_k=60):

    q_emb = dense_model.encode(query,convert_to_tensor=True,normalize_embeddings=True)

    dense_scores = cos_sim(q_emb, corpus_embeddings)[0]
    dense_idx = torch.topk(dense_scores, dense_k).indices.tolist()

    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i],reverse=True)[:bm25_k]

    rrf_scores = defaultdict(float)

    for rank, idx in enumerate(dense_idx, start=1):
        rrf_scores[idx] += 1.0 / (rrf_k + rank)

    for rank, idx in enumerate(bm25_idx, start=1):
        rrf_scores[idx] += 1.0 / (rrf_k + rank)

    fused_idx = sorted(
        rrf_scores.keys(),
        key=lambda i: rrf_scores[i],
        reverse=True
    )

    return [corpus_texts[i] for i in fused_idx]


def batched_ce_predict(reranker, pairs, batch_size=64):
  scores = []
  reranker.model.eval()
  with torch.no_grad():
      def collate_fn(batch):
          return batch

      for batch in DataLoader(pairs, batch_size=batch_size, collate_fn=collate_fn):
          scores.extend(reranker.predict(batch))
  return scores


def corpus_level_rrf_rerank_eval(dense_model,reranker,triplets,corpus_embeddings,corpus_texts,
                                bm25, ks=[5, 10, 20], dense_k=200, bm25_k=200, rrf_k=60,max_candidates=200):

    N = len(triplets)

    mrr = 0.0
    recall = {k: 0.0 for k in ks}
    precision = {k: 0.0 for k in ks}
    ndcg = {k: 0.0 for k in ks}

    retrieval_success = 0
    mrr_cond = 0.0

    for i, t in enumerate(triplets):
        if i % 50 == 0:
            print(f"Processed {i}/{N} queries")

        query = t["query"]
        gold_text = t["positive"]

        candidates = rrf_retrieve(
            query=query,
            dense_model=dense_model,
            corpus_embeddings=corpus_embeddings,
            corpus_texts=corpus_texts,
            bm25=bm25,
            dense_k=dense_k,
            bm25_k=bm25_k,
            rrf_k=rrf_k
        )

        candidates = candidates[:max_candidates]

        # print("Num candidates:", len(candidates))

        if gold_text not in candidates:
            continue

        retrieval_success += 1

        pairs = [[query, c] for c in candidates]

        ce_scores = batched_ce_predict(reranker, pairs, batch_size=64)

        reranked = sorted(
            zip(candidates, ce_scores),
            key=lambda x: -x[1]
        )

        ranked_texts = [c for c, _ in reranked]
        rank = ranked_texts.index(gold_text) + 1

        mrr += 1 / rank
        mrr_cond += 1 / rank

        for k in ks:
            if rank <= k:
                recall[k] += 1
                precision[k] += 1 / k
                ndcg[k] += 1 / math.log2(rank + 1)

    results = {
        "Num_Queries": N,
        "RRF_Retrieval_Recall": retrieval_success / N,
        "MRR": mrr / N,
        "MRR_given_retrieval": mrr_cond / max(retrieval_success, 1)
    }

    for k in ks:
        results[f"Recall@{k}"] = recall[k] / N
        results[f"Precision@{k}"] = precision[k] / N
        results[f"nDCG@{k}"] = ndcg[k] / N

    return results

!unzip best_finetuned_dense_encoder.zip -d dense_encoder
!unzip ce_finder_best.zip -d ce_finder_best
from sentence_transformers import SentenceTransformer, CrossEncoder

dense_model = SentenceTransformer("dense_encoder/models/finder_dense_encoder_best/")
dense_model.max_seq_length = 256

reranker = CrossEncoder("ce_finder_best/cross_encoder_finder_best/",max_length=512)


def load_finder_triplets(path):
    triplets = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            triplets.append({
                "query": item["query"],
                "positive": item["positive"]["text"],
                "negatives": [n["text"] for n in item["negatives"]]
            })
    return triplets

triplets = load_finder_triplets("finder_triplets_optimized.jsonl")
print(f"Loaded {len(triplets)} evaluation queries")
with open("fine_tuned_sec_embeddings.pkl","rb") as f:
    corpus_embeddings = pickle.load(f)
device = "cuda" if torch.cuda.is_available() else "cpu"

dense_model = dense_model.to(device)
reranker.model.to(device)

corpus_embeddings = corpus_embeddings.to(device)
results = corpus_level_rrf_rerank_eval(
    dense_model,
    reranker,
    triplets,
    corpus_embeddings,
    corpus_texts,
    bm25,
    ks=[5, 10, 20],
    dense_k=200,
    bm25_k=200,
    rrf_k=60
)

Found 293 companies in FINdER dataset: {'AMAT', 'AWK', 'TROW', 'AVGO', 'CMCSA', 'NVDA', 'KKR', 'LRCX', 'BX', 'XYL', 'NWSA', 'GEN', 'TAP', 'AMD', 'NFLX', 'CF', 'MCO', 'SPG', 'IVZ', 'VRSK', 'REGN', 'INCY', 'BIIB', 'CAH', 'NTAP', 'JKHY', 'CVX', 'PNC', 'AJG', 'ABBV', 'CME', 'IDXX', 'PEP', 'ED', 'DE', 'TECH', 'RTX', 'DLTR', 'MTB', 'NXPI', 'ADI', 'AOS', 'COP', 'ES', 'NEM', 'TSCO', 'META', 'AAPL', 'LIN', 'GILD', 'AVY', 'ADSK', 'CEG', 'SJM', 'ISRG', 'EXPE', 'NVR', 'SYK', 'WMT', 'CDNS', 'WBD', 'GM', 'STLD', 'BEN', 'NSC', 'OXY', 'ULTA', 'NDAQ', 'POOL', 'BF.B', 'TMUS', 'SPGI', 'MMC', 'BKNG', 'DG', 'BAC', 'CMG', 'TXN', 'XEL', 'LNT', 'LVS', 'LYB', 'AMGN', 'EPAM', 'VRTX', 'BXP', 'PANW', 'MNST', 'BKR', 'RJF', 'MS', 'MLM', 'GS', 'MRK', 'FTNT', 'AMCR', 'ETR', 'ADBE', 'WYNN', 'RF', 'DPZ,', 'LKQ', 'BAX', 'IEX', 'DLR', 'KVUE', 'HST', 'ADP', 'TTWO', 'AME', 'PRU', 'CHTR', 'TSLA', 'CDW', 'PFG', 'QCOM', 'PSA', 'AXON', 'TRMB', 'PNW', 'ATO', 'GOOGL', 'NOC', 'KEYS', 'STX', 'CBOE', 'CCI', 'MOS', 'TGT', 'DXCM', 'A

In [ ]:
results

{'Num_Queries': 3439,
 'RRF_Retrieval_Recall': 0.6304158185519046,
 'MRR': 0.20584464062046598,
 'MRR_given_retrieval': 0.3265220106521137,
 'Recall@5': 0.2471648735097412,
 'Precision@5': 0.04943297470194792,
 'nDCG@5': 0.20296042677477408,
 'Recall@10': 0.3041581855190462,
 'Precision@10': 0.030415818551904138,
 'nDCG@10': 0.2212026613632197,
 'Recall@20': 0.3788892119802268,
 'Precision@20': 0.01894446059901089,
 'nDCG@20': 0.23997173866718421}